---
# COSC2753 | Machine Learning

## Task 1 — Member 1: Article-Type Classification
---

# 1. Introduction

This notebook is the complete Task 1 workflow: target-specific EDA, a majority baseline, an independently initialized CNN, validation-based selection, test evaluation, checkpoint saving, and inference. Article type is a severe long-tail problem, so macro F1 is primary and top-3 accuracy is supporting evidence.

# 2. Library Imports and Setup

In [ ]:
from copy import deepcopy
from pathlib import Path
import json, sys, time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, top_k_accuracy_score)
from skimage.feature import hog
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from preprocessing import IMAGE_SIZE, NORMALISATION_PATH, SEED, seed_everything, task_frame

seed_everything(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

# 3. Load the Frozen Data

In [ ]:
TARGET = 'articleType'
train_df = task_frame(TARGET, 'train')
validation_df = task_frame(TARGET, 'validation')
test_df = task_frame(TARGET, 'test')
with NORMALISATION_PATH.open(encoding='utf-8') as handle:
    normalisation = json.load(handle)
labels = sorted(train_df[TARGET].unique())
label_to_index = {label: index for index, label in enumerate(labels)}
len(train_df), len(validation_df), len(test_df), len(labels)

# 4. Target-Specific EDA

In [ ]:
counts = train_df[TARGET].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=counts.head(30).values, y=counts.head(30).index, ax=axes[0])
axes[0].set_title('Thirty largest article-type classes')
axes[1].hist(counts.values, bins=30)
axes[1].set_title('Distribution of per-class support')
axes[1].set_xlabel('Training images per class')
plt.tight_layout()
counts.describe()

## 4.1 Observations

Define head, medium, and tail support thresholds here. Check whether every validation/test label exists in training and discuss unsupported or extremely rare classes. Inspect visually adjacent labels before choosing loss weighting.

In [ ]:
{
    'validation_only': sorted(set(validation_df[TARGET]) - set(labels)),
    'test_only': sorted(set(test_df[TARGET]) - set(labels)),
    'classes_below_10': int((counts < 10).sum()),
    'classes_below_50': int((counts < 50).sum()),
}

# 5. Majority Baseline

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(np.zeros((len(train_df), 1)), train_df[TARGET])
dummy_predictions = dummy.predict(np.zeros((len(validation_df), 1)))
{
    'accuracy': accuracy_score(validation_df[TARGET], dummy_predictions),
    'macro_f1': f1_score(validation_df[TARGET], dummy_predictions, average='macro', zero_division=0),
}

# 6. HOG + HSV Classical Baseline

HOG represents garment shape while HSV histograms represent colour. The class-weighted linear model tests how far transparent handcrafted features can go before deep learning.

In [ ]:
def handcrafted_feature(path):
    with Image.open(path) as source:
        image = source.convert('RGB').resize((48, 64))
    array = np.asarray(image, dtype=np.float32) / 255.0
    gray = array.mean(axis=2)
    shape = hog(gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
    hsv = np.asarray(image.convert('HSV'), dtype=np.float32) / 255.0
    colour = np.concatenate([np.histogram(hsv[..., channel], bins=8, range=(0, 1), density=True)[0] for channel in range(3)])
    return np.concatenate([shape, colour]).astype(np.float32)

def feature_matrix(frame):
    return np.vstack([handcrafted_feature(path) for path in frame.image_path])

train_features = feature_matrix(train_df)
validation_features = feature_matrix(validation_df)
linear_baseline = LogisticRegression(class_weight='balanced', max_iter=300, n_jobs=-1)
linear_baseline.fit(train_features, train_df[TARGET])
linear_predictions = linear_baseline.predict(validation_features)
{
    'accuracy': accuracy_score(validation_df[TARGET], linear_predictions),
    'macro_f1': f1_score(validation_df[TARGET], linear_predictions, average='macro', zero_division=0),
}

# 7. Image Pipeline

Augmentation is applied only to training images. Validation and test images use deterministic RGB conversion, resize, tensor conversion, and training-fitted normalisation.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize(normalisation['mean'], normalisation['std']),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])),
    transforms.ToTensor(),
    transforms.Normalize(normalisation['mean'], normalisation['std']),
])

class FashionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, label_to_index[row[TARGET]]

train_loader = DataLoader(FashionDataset(train_df, train_transform), batch_size=64, shuffle=True, num_workers=0)
validation_loader = DataLoader(FashionDataset(validation_df, evaluation_transform), batch_size=128, num_workers=0)
test_loader = DataLoader(FashionDataset(test_df, evaluation_transform), batch_size=128, num_workers=0)

# 8. Compact Residual CNN From Scratch

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(output_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if input_channels == output_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 1, stride, bias=False), nn.BatchNorm2d(output_channels),
        )
    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.relu(self.bn1(self.conv1(inputs)))
        outputs = self.bn2(self.conv2(outputs))
        return self.relu(outputs + residual)

class CompactCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32), nn.MaxPool2d(2),
            ResidualBlock(32, 64), nn.MaxPool2d(2),
            ResidualBlock(64, 128), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, inputs):
        return self.classifier(self.features(inputs))

model = CompactCNN(len(labels)).to(DEVICE)
sum(parameter.numel() for parameter in model.parameters())

# 9. Class-Balanced Training

In [ ]:
# Run once with 'ordinary' and once with 'class_balanced', resetting the model and seed each time.
LOSS_MODE = 'class_balanced'
class_counts = train_df[TARGET].value_counts().reindex(labels).values
beta = 0.9999
effective_number = 1.0 - np.power(beta, class_counts)
class_weights = (1.0 - beta) / effective_number
class_weights = class_weights / class_weights.mean()
loss_weights = None if LOSS_MODE == 'ordinary' else torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, true_labels, predicted_labels = [], [], []
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        losses.append(loss.item() * len(targets))
        true_labels.extend(targets.cpu().tolist())
        predicted_labels.extend(logits.argmax(1).cpu().tolist())
    return {
        'loss': sum(losses) / len(loader.dataset),
        'accuracy': accuracy_score(true_labels, predicted_labels),
        'macro_f1': f1_score(true_labels, predicted_labels, average='macro', zero_division=0),
    }

In [ ]:
history, best_state, best_f1, patience = [], None, -1.0, 0
for epoch in range(1, 41):
    started = time.perf_counter()
    train_metrics = run_epoch(model, train_loader, optimizer)
    validation_metrics = run_epoch(model, validation_loader)
    history.append({'epoch': epoch, 'loss_mode': LOSS_MODE, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'validation_{k}': v for k, v in validation_metrics.items()}})
    print(epoch, validation_metrics, f'{time.perf_counter() - started:.1f}s')
    if validation_metrics['macro_f1'] > best_f1:
        best_f1, best_state, patience = validation_metrics['macro_f1'], deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= 7:
            break
model.load_state_dict(best_state)
history = pd.DataFrame(history)

In [ ]:
history.plot(x='epoch', y=['train_macro_f1', 'validation_macro_f1'], figsize=(8, 4), title='Article-type learning curve')
plt.show()

## 8.1 Training analysis

Interpret convergence, the train-validation gap, and the epoch selected by validation macro F1. Compare ordinary and weighted loss as controlled runs before finalizing this cell's configuration.

# 10. Final Evaluation and Failure Analysis

Run this section only after model choices are frozen.

In [ ]:
@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    probabilities, truth = [], []
    for images, targets in loader:
        probabilities.append(model(images.to(DEVICE)).softmax(1).cpu().numpy())
        truth.extend(targets.tolist())
    return np.vstack(probabilities), np.asarray(truth)

test_probabilities, test_truth = predict_loader(model, test_loader)
test_predictions = test_probabilities.argmax(1)
test_metrics = {
    'accuracy': accuracy_score(test_truth, test_predictions),
    'macro_f1': f1_score(test_truth, test_predictions, average='macro', zero_division=0),
    'top_3_accuracy': top_k_accuracy_score(test_truth, test_probabilities, k=3, labels=np.arange(len(labels))),
}
test_metrics

In [ ]:
print(classification_report(
    test_truth, test_predictions, labels=np.arange(len(labels)), target_names=labels, zero_division=0,
))

In [ ]:
largest_labels = counts.head(20).index
largest_indices = [label_to_index[label] for label in largest_labels]
selected = np.isin(test_truth, largest_indices)
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(
    test_truth[selected], test_predictions[selected], labels=largest_indices,
    display_labels=largest_labels, normalize='true', xticks_rotation=90, ax=ax,
)
plt.title('Frequent-class normalized confusion matrix')
plt.show()

## 9.1 Failure analysis

Report head/medium/tail macro F1, the most frequent confusion pairs, high-confidence mistakes, robustness to mild brightness/blur changes, inference latency, and checkpoint size. Do not claim reliable performance for labels with inadequate test support.

# 11. Save the Selected Model

In [ ]:
checkpoint_path = ROOT / 'models' / 'article_type_model.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'target': TARGET, 'labels': labels, 'state_dict': model.cpu().state_dict(),
    'mean': normalisation['mean'], 'std': normalisation['std'],
    'image_size': list(IMAGE_SIZE), 'dropout': 0.2, 'selection_metric': 'validation_macro_f1',
    'best_validation_macro_f1': best_f1, 'test_metrics': test_metrics, 'seed': SEED,
}, checkpoint_path)
history.to_csv(ROOT / 'models' / 'article_type_history.csv', index=False)
checkpoint_path

# 12. Final Prediction Check

In [ ]:
# Run from the repository root after saving:
# python scripts/task1_article_type_classification.py path/to/image.jpg

# 13. Ultimate Judgement and Conclusion

Replace this prompt with the measured comparison among majority, HOG+HSV, ordinary-loss residual CNN, and class-balanced residual CNN; include the selected epoch, macro/top-3 results, major confusions, efficiency, limitations, and why this checkpoint is suitable for integration.